# CD1 · Aula 08 — Laboratório: busca em grade (`GridSearchCV`)

Até aqui você aprendeu a **medir** um modelo com honestidade: treino e teste, a régua certa, a validação cruzada. Hoje a pergunta é outra: **qual configuração do modelo usar?** O `C` da logística e o `class_weight` são botões que você gira **antes** do `fit`: os hiperparâmetros.

A busca em grade testa **todas** as combinações de uma lista de valores e deixa a validação cruzada escolher. Você vai praticar:

1. parâmetro × hiperparâmetro, e por que a escolha acontece na validação, nunca no teste;
2. o `GridSearchCV` sobre um `Pipeline`, com o prefixo `clf__`, o `scoring` de classe rara e o `cv` estratificado;
3. ler `best_params_`, `best_score_` e `cv_results_`, e reconhecer o platô;
4. o custo: combinações × dobras, e por que a grade explode;
5. o teste tocado uma única vez;
6. como o `scoring` decide quem vence.

**Como o lab funciona.** Cada exercício traz primeiro um **exemplo resolvido**, numa versão menor do mesmo problema. Rode, leia, e depois resolva o **"Agora é com você"**.

**Dados:** `5_musicas.csv`, que está **nesta mesma pasta**. Cada linha é uma faixa; o alvo `hit` marca os sucessos (cerca de 12%).

## Parte 0 — Ambiente e dados

Carregamos as músicas, tiramos a `popularidade` (ela praticamente define o hit: seria vazamento), transformamos o gênero em colunas e separamos 30% para teste. Rode esta célula antes de tudo.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

mus = pd.read_csv("5_musicas.csv")                 # o CSV está na mesma pasta deste notebook
y = mus["hit"]                                      # alvo: 1 = hit (cerca de 12% das faixas)
X = mus.drop(columns=["id", "titulo", "artista", "popularidade", "hit"])  # popularidade = vazamento
X = pd.get_dummies(X, columns=["genero"])           # gênero vira colunas 0/1
print("X:", X.shape, "| proporção de hits:", round(y.mean(), 3))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print("treino:", len(X_tr), "| teste:", len(X_te), "| hits no teste:", int(y_te.sum()))
print("o teste fica GUARDADO até o Exercício 5")

pipe = Pipeline([("esc", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=2000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("passos do pipeline:", list(pipe.named_steps))

## Exercício 1 — Parâmetro × hiperparâmetro

Treine o `pipe` (regressão logística com `C = 1`) no treino e imprima: o valor de `C` e de `class_weight` e o formato de `coef_`. Depois, no campo de texto, classifique como **parâmetro** ou **hiperparâmetro**: (a) o coeficiente de `danceability`; (b) o `C`; (c) o `class_weight`; (d) o corte que uma árvore faz em cada pergunta. E responda: por que escolher o `C` olhando o **teste** seria um erro?

**Exemplo antes de começar.** a mesma pergunta com uma árvore de decisão (Aula 01). A profundidade máxima, `max_depth = 3`, **você** escolheu antes do `fit`: é hiperparâmetro. Qual variável a primeira pergunta usa e em que valor ela corta, a árvore **aprendeu** dos dados: são parâmetros.

In [ ]:
arv = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
print("max_depth (hiperparâmetro, VOCÊ escolheu):", arv.max_depth)
print("1ª pergunta (parâmetro, APRENDIDO)       :",
      X.columns[arv.tree_.feature[0]], "<=", round(arv.tree_.threshold[0], 3))

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
pipe.fit(...)
clf = pipe.named_steps["clf"]
print("C:", ...)
print("class_weight:", ...)
print("formato de coef_:", ...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 2 — `GridSearchCV` sobre o `Pipeline`

Rode a busca da grade da aula: `clf__C` em `[0.01, 0.1, 1, 10, 100]` e `clf__class_weight` em `[None, "balanced"]`, com `scoring="average_precision"` e `cv=cv`. Guarde a grade na variável `grade` e a busca em `busca`.

**Exemplo antes de começar.** uma busca minúscula, só com dois valores de `C`, para ver o formato. O `GridSearchCV` recebe o Pipeline, a grade, a métrica e as dobras. O prefixo `clf__` diz que o `C` pertence ao passo `"clf"` do Pipeline.

In [ ]:
mini = GridSearchCV(pipe, {"clf__C": [0.1, 10]}, scoring="average_precision", cv=cv)
mini.fit(X_tr, y_tr)
print("mini-busca concluída:", len(mini.cv_results_["params"]), "combinações testadas")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
grade = {"clf__C": [...], "clf__class_weight": [...]}
busca = GridSearchCV(pipe, ..., scoring=..., cv=...)
busca.fit(X_tr, y_tr)
print("busca concluída:", len(busca.cv_results_["params"]), "combinações testadas")

## Exercício 3 — Lendo o resultado e achando o platô

Imprima o `best_params_` e o `best_score_` da `busca` e a tabela do `cv_results_` ordenada por `rank_test_score`, com as colunas `param_clf__C`, `param_clf__class_weight`, `mean_test_score` e `std_test_score`. Depois conte quantas das 10 combinações ficam a **menos de um desvio** da campeã (o platô da Aula 07).

**Exemplo antes de começar.** as mesmas leituras na mini-busca. `best_params_` é a combinação vencedora; `best_score_` é a média de validação dela (validação, não teste); `cv_results_` tem uma linha por combinação. Repare: a diferença entre as duas é bem menor que o desvio entre dobras.

In [ ]:
print("best_params_:", mini.best_params_)
print(f"best_score_: {mini.best_score_:.4f}   (média das 5 dobras de validação)")
tab_mini = pd.DataFrame(mini.cv_results_)
print(tab_mini[["param_clf__C", "mean_test_score", "std_test_score"]].to_string(index=False))

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
print("best_params_:", ...)
print("best_score_:", ...)
tab = pd.DataFrame(busca.cv_results_).sort_values(...)
tab["param_clf__class_weight"] = tab["param_clf__class_weight"].fillna("None")  # None aparece como NaN
print(tab[[...]].to_string(index=False))

campea = tab.iloc[0]
dentro = ...
print("combinações a menos de 1 desvio da campeã:", dentro, "de", len(tab))

*Sua resposta:*

_(escreva aqui)_

## Exercício 4 — O custo: a grade explode

Calcule o número de treinos da `busca` (`combinações × dobras`) e meça o tempo dela com `time.perf_counter()`. Depois faça a conta para a floresta aleatória do próximo lab, com 5 hiperparâmetros e 4 · 5 · 4 · 4 · 4 valores, também com 5 dobras. Quantas vezes maior é?

**Exemplo antes de começar.** a conta da mini-busca: 2 valores de `C` × 5 dobras = 10 treinos. O próprio objeto confirma: `cv_results_` tem uma linha por combinação, e `cv.get_n_splits()` dá o número de dobras. (O `GridSearchCV` ainda faz **mais 1** treino no fim, com a vencedora em todo o treino: é o `best_estimator_`.)

In [ ]:
n_comb   = len(mini.cv_results_["params"])
n_dobras = cv.get_n_splits()
t0 = time.perf_counter()
GridSearchCV(pipe, {"clf__C": [0.1, 10]}, scoring="average_precision", cv=cv).fit(X_tr, y_tr)
t_mini = time.perf_counter() - t0
print(f"{n_comb} combinações × {n_dobras} dobras = {n_comb * n_dobras} treinos  ->  {t_mini:.2f} s")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
n_comb   = ...
n_ajustes = ...
t0 = time.perf_counter()
GridSearchCV(...).fit(X_tr, y_tr)
t_busca = time.perf_counter() - t0
print(...)

n_floresta = ...
print(...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 5 — O teste, uma única vez

Agora (e só agora) toque o teste. Pegue o `busca.best_estimator_`, gere as probabilidades no `X_te` e calcule a **AP** e a **ROC AUC**. Compare a AP de teste com o `best_score_`.

**Exemplo antes de começar.** a AP na mão, com 5 músicas (Aula 03). Ordene pela probabilidade e, em cada hit, anote a precisão até ali. A AP é a média dessas precisões. Aqui: o 1º da lista é hit (precisão 1/1); o 2º não; o 3º é hit (precisão 2/3). AP = (1 + 0,667) / 2 = 0,833.

In [ ]:
y_mao = [0, 1, 0, 1, 0]
p_mao = [0.10, 0.90, 0.40, 0.35, 0.20]
print("AP na mão :", round((1 + 2/3) / 2, 3))
print("AP sklearn:", round(average_precision_score(y_mao, p_mao), 3))

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
melhor = ...                          # já vem treinado em todo o treino
proba_te = melhor.predict_proba(...)[:, 1]
ap_te  = ...
auc_te = ...
print(...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 6 — O scoring decide quem vence

Rode a mesma `grade` três vezes, com `scoring` igual a `"average_precision"`, `"accuracy"` e `"f1"`, e monte uma tabela com uma linha por combinação e uma coluna por métrica (a média de validação). Depois olhe a linha `C = 0.01`, `class_weight = None`: o que a acurácia diz dela, e o que o F1 diz?

**Exemplo antes de começar.** o "chute" que prevê sempre "não é hit". Ele acerta 88% (acurácia alta) e não encontra nenhum hit. A AP dele é a própria proporção de hits: a régua mostra que ele não sabe nada.

In [ ]:
chute = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(f"acurácia do chute: {(chute.predict(X_tr) == y_tr).mean():.3f}   <- alta e enganosa")
print(f"AP do chute      : {average_precision_score(y_tr, chute.predict_proba(X_tr)[:, 1]):.3f}"
      "   <- = proporção de hits")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
medias = {}
for metrica in ["average_precision", "accuracy", "f1"]:
    g = GridSearchCV(pipe, grade, scoring=..., cv=cv).fit(X_tr, y_tr)
    medias[metrica] = ...
comp = pd.DataFrame(medias)
comp.index = [f"C={p['clf__C']}, cw={p['clf__class_weight']}" for p in g.cv_results_["params"]]
print(comp.round(3))

*Sua resposta:*

_(escreva aqui)_

## Exercício 7 — Três conclusões

Escreva três conclusões do laboratório. Sugestões: (1) o que é ajustar hiperparâmetro e onde essa escolha acontece; (2) o que o `Pipeline` e o prefixo `clf__` fazem dentro da busca; (3) quanto a busca custa e qual número se reporta no fim.

*Suas conclusões:*

_(escreva aqui)_

---
## Fecho

| o quê | para que serve |
|---|---|
| `best_params_` | a combinação escolhida |
| `best_score_` | a média de validação da escolhida: serve para **escolher**, não para reportar |
| `cv_results_` | a tabela inteira: mostra o platô e o desvio entre dobras |
| `best_estimator_` | o Pipeline já treinado com a escolhida: é ele que vai ao teste, uma vez |

**Na próxima aula:** a floresta do Exercício 4 teria 6.400 treinos. Em vez de testar todas as combinações, vamos **sortear** algumas: a busca aleatória (`RandomizedSearchCV`).